In [9]:
import pandas as pd
import json
import sqlite3
import numpy as np
from scipy.spatial.distance import pdist, squareform

In [10]:
spellings_path = "data/toponym_spellings_extended.txt"

In [11]:
with open(spellings_path, "r") as fp:
    normalization_dict = {}
    grouped_spellings = []
    for s in fp.readlines():
        s = s.replace("\n", "")
        val_key_split = s.split("|")
        val = val_key_split[0]
        keys = val_key_split[-1].split("@")
        for key in keys:
            normalization_dict[key] = val

        grouped_spellings.append(
            set(keys)
        )

In [12]:
with open("outputs/json/db_toponyms.json", "w") as fp:
    json.dump(normalization_dict, fp, ensure_ascii=False, indent=4)

## Adding coordinates

In [13]:
connection = sqlite3.connect("data/all_hiski_records.sqlite3")
cursor = connection.cursor()

query = """
    SELECT *
    FROM parish;
"""

cursor.execute(query)

parish_id_to_info = {}
parish_name_to_info = {}
for row in cursor.fetchall():
    parish_id_to_info[row[0]] = {
        "name": row[2],
        "namese": row[3],
        "lon": row[6],
        "lat": row[7],
    }

    parish_name_to_info[row[2]] = {
        "parish_id": row[0],
        "namese": row[3],
        "lon": row[6],
        "lat": row[7],
    }

For each group, assign a group_id and coordinates. The coordinates are for now set to be the average of all coordinates in the group:

In [14]:
spelling_groups = {i: g for i, g in enumerate(grouped_spellings)}
group_coords = {}
for group_id, group in spelling_groups.items():
    group_infos = []
    for s in group:
        info = parish_name_to_info.get(s, None)
        if info:
            group_infos.append(info)
    coords = np.array([(x["lon"], x["lat"]) for x in group_infos])
    if coords.any():
        group_coords[group_id] = np.mean(coords[:, 0]), np.mean(coords[:, 1])
    else:
        group_coords[group_id] = None

In [15]:
spelling_to_group_id = {}
for g_id, group in spelling_groups.items():
    for spelling in group:
        spelling_to_group_id[spelling] = g_id

In [16]:
group_coords_dropped_none = {k: v for k, v in group_coords.items() if v}

Now we can gather coordinates and normalized form for each spelling.

In [17]:
default = (None, None)
normalization_dict_with_coords = {
    k: {
        "normalized": v,
        "e": group_coords_dropped_none.get(
            spelling_to_group_id[k], default
        )[0],
        "n": group_coords_dropped_none.get(
            spelling_to_group_id[k], default
        )[1]
    }
    for k, v in normalization_dict.items()
}

In [18]:
with open("outputs/json/db_toponyms_with_coordinates.json", "w") as fp:
    json.dump(
        normalization_dict_with_coords, fp,
        ensure_ascii=False
    )

In [19]:
with open("outputs/json/db_spelling_groups.json", "w") as fp:
    json.dump(
        {k: tuple(v) for k, v in spelling_groups.items()}, fp,
        ensure_ascii=False, indent=4
    )
with open("outputs/json/db_spelling_to_group_id.json", "w") as fp:
    json.dump(
        spelling_to_group_id, fp,
        ensure_ascii=False, indent=4
    )